In [1]:
import heapq
import copy

# Define the goal state
GOAL_STATE = ((1, 2, 3),
              (8, 0, 4),
              (7, 6, 5))

class PuzzleNode:
    def __init__(self, state, parent=None, move=None, g=0, h=0):
        self.state = state  # 2D tuple representation of the board
        self.parent = parent
        self.move = move
        self.g = g  # Path cost from start to current node
        self.h = h  # Heuristic cost estimate to goal
        self.f = g + h  # Total estimated cost

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.state[r][c] == 0:
                    return r, c

    def generate_actions(self):
        actions = []
        r, c = self.get_blank_pos()
        moves = {
            'Up': (r - 1, c),
            'Down': (r + 1, c),
            'Left': (r, c - 1),
            'Right': (r, c + 1)
        }
        for move_name, (nr, nc) in moves.items():
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Create new state
                new_state = [list(row) for row in self.state]
                new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
                new_state_tuple = tuple(tuple(row) for row in new_state)
                actions.append((new_state_tuple, move_name))
        return actions

def a_star_search(start_state, heuristic_fn, goal_state=GOAL_STATE):
    start_node = PuzzleNode(start_state, g=0, h=heuristic_fn(start_state))
    open_list = []
    heapq.heappush(open_list, start_node)
    closed_list = set()
    nodes_expanded = 0

    while open_list:
        current_node = heapq.heappop(open_list)
        nodes_expanded += 1

        if current_node.state == goal_state:
            # Reconstruct path
            path = []
            curr = current_node
            while curr.parent is not None:
                path.append((curr.move, curr.state))
                curr = curr.parent
            path.reverse()
            return path, nodes_expanded

        closed_list.add(current_node.state)

        for next_state, action in current_node.generate_actions():
            if next_state in closed_list:
                continue

            g_cost = current_node.g + 1
            h_cost = heuristic_fn(next_state)
            child_node = PuzzleNode(next_state, parent=current_node, move=action, g=g_cost, h=h_cost)
            heapq.heappush(open_list, child_node)

    return None, nodes_expanded

def print_path_details(path, start_state, heuristic_fn, name):
    print(f"=== {name} Heuristic Path ===\n")
    curr_state = start_state
    g = 0
    h = heuristic_fn(curr_state)
    f = g + h

    print("Initial State:")
    for i, row in enumerate(curr_state):
        row_str = " ".join(str(val) if val != 0 else "_" for val in row)
        if i == 1:
            print(f"  [ {row_str} ]    g = {g}, h = {h}, f = {f}")
        else:
            print(f"  [ {row_str} ]")
    print("\n" + "-"*50 + "\n")

    for step, (move, next_state) in enumerate(path, 1):
        g = step
        h = heuristic_fn(next_state)
        f = g + h
        print(f"Step {step} (Move {move}):")
        for i, row in enumerate(next_state):
            row_str = " ".join(str(val) if val != 0 else "_" for val in row)
            if i == 1:
                print(f"  [ {row_str} ]    g = {g}, h = {h}, f = {f}")
            else:
                print(f"  [ {row_str} ]")
        print()
    print("="*50 + "\n")

In [2]:
def misplaced_tiles_heuristic(state, goal=GOAL_STATE):
    misplaced = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0 and val != goal[r][c]:
                misplaced += 1
    return misplaced

In [3]:
def manhattan_distance_heuristic(state, goal=GOAL_STATE):
    # Create a mapping of value -> (row, col) in the goal state
    goal_map = {}
    for r in range(3):
        for c in range(3):
            goal_map[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                target_r, target_c = goal_map[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

In [4]:
start_state = ((2, 8, 3),
               (1, 6, 4),
               (7, 0, 5))

path_misplaced, _ = a_star_search(start_state, misplaced_tiles_heuristic)
path_manhattan, _ = a_star_search(start_state, manhattan_distance_heuristic)

print_path_details(path_misplaced, start_state, misplaced_tiles_heuristic, "Misplaced Tiles")
print_path_details(path_manhattan, start_state, manhattan_distance_heuristic, "Manhattan Distance")

=== Misplaced Tiles Heuristic Path ===

Initial State:
  [ 2 8 3 ]
  [ 1 6 4 ]    g = 0, h = 4, f = 4
  [ 7 _ 5 ]

--------------------------------------------------

Step 1 (Move Up):
  [ 2 8 3 ]
  [ 1 _ 4 ]    g = 1, h = 3, f = 4
  [ 7 6 5 ]

Step 2 (Move Up):
  [ 2 _ 3 ]
  [ 1 8 4 ]    g = 2, h = 3, f = 5
  [ 7 6 5 ]

Step 3 (Move Left):
  [ _ 2 3 ]
  [ 1 8 4 ]    g = 3, h = 2, f = 5
  [ 7 6 5 ]

Step 4 (Move Down):
  [ 1 2 3 ]
  [ _ 8 4 ]    g = 4, h = 1, f = 5
  [ 7 6 5 ]

Step 5 (Move Right):
  [ 1 2 3 ]
  [ 8 _ 4 ]    g = 5, h = 0, f = 5
  [ 7 6 5 ]


=== Manhattan Distance Heuristic Path ===

Initial State:
  [ 2 8 3 ]
  [ 1 6 4 ]    g = 0, h = 5, f = 5
  [ 7 _ 5 ]

--------------------------------------------------

Step 1 (Move Up):
  [ 2 8 3 ]
  [ 1 _ 4 ]    g = 1, h = 4, f = 5
  [ 7 6 5 ]

Step 2 (Move Up):
  [ 2 _ 3 ]
  [ 1 8 4 ]    g = 2, h = 3, f = 5
  [ 7 6 5 ]

Step 3 (Move Left):
  [ _ 2 3 ]
  [ 1 8 4 ]    g = 3, h = 2, f = 5
  [ 7 6 5 ]

Step 4 (Move Down):
  [ 1 2 3